In [1]:
import numpy as np
from random import uniform as _uniform, choice as _choice
from numpy.random import random, uniform
from numba import njit

def _make_dist_matrix(X, measure, symmetric=True):
    """Generate a distance matrix.

    Parameters
    ----------
    X : array_like (N, k)
        Dataset with nodes' features.
        One row is one node.
    measure : callable
        Measure function that takes two main arguments which are
        feature vectors for two nodes.
    symmetric : bool
        Is the measure function symmetric in the two main arguments.

    Returns
    -------
    (N, N) array_like
        Edge formation probability matrix.
    """
    N = X.shape[0]
    D = np.zeros((N, N))
    if symmetric:
        for i in range(N):
            for j in range(i):
                D[i, j] = D[j, i] = measure(X[i:i+1], X[j:j+1])
    else:
        for i in range(N):
            for j in range(N):
                D[i, j] = measure(X[i:i+1], X[j:j+1])
    return D

make_dist_matrix = njit(_make_dist_matrix)

def make_adjacency_matrix(P, directed=False):
    """Generate adjacency matrix from edge formation probabilities.

    Parameters
    ----------
    P : (N, N) array_like
        Edge formation probability matrix.
    directed : bool
        Should network be directed.
    """
    # pylint: disable=no-member
    if directed:
        A = np.where(uniform(0, 1, P.shape) <= P, 1, 0)
        A = A.astype(int)
        np.fill_diagonal(A, 0)
    else:
        A = np.zeros_like(P, dtype=int)
        A = _am_undirected_nb(P, A)
    return A

def rewire_edges(A, p=0.01, directed=False, copy=False):
    """Randomly rewire edges in an adjacency matrix with given probability.

    Parameters
    ----------
    A : (N, N) array_like
        An adjacency matrix.
    p : float
        Rewiring probability.
    directed : bool
        Is the graph directed.
    copy : bool
        Should copy of the adjacency array be returned.
    """
    if copy:
        A = A.copy()
    E = get_edgelist(A, directed=directed)
    loop = range(0, E.shape[0]) if directed else range(0, E.shape[0], 2)
    for u in loop:
        rand = _uniform(0, 1)
        if rand <= p:
            i, j = E[u, :2]
            if not directed and rand <= p/2:
                new_i = j
            else:
                new_i = i
            idx = np.nonzero(np.where(A[new_i, :] == 0, 1, 0))[0]
            idx = idx[idx != new_i]
            if idx.size == 0:
                continue
            new_j = _choice(idx)
            A[i, j] = 0
            A[new_i, new_j] = 1
            if not directed:
                A[j, i] = 0
                A[new_j, new_i] = 1
    return A

@njit
def random_integer(p):
    c = np.cumsum(p)
    x = uniform(0, c[-1])
    return np.where(c >= x)[0][0]

@njit
def euclidean_dist(u, v, normalized=False):
    return Lp_dist(u, v, p=2, normalized=normalized)

In [2]:
"""Social distance attachment network model."""
# pylint: disable=R0902
# pylint: disable=arguments-differ,too-many-locals
from collections import OrderedDict
from itertools import product
import numpy as np
from scipy.sparse import csr_matrix
from scipy.optimize import toms748

class SDA:
    """Social distance attachment network model.

    Attributes
    ----------
    P : (N, N) array_like
        Edge formation probability matrix.
    k : float
        Expected average node degree.
    b : float
        Characteristic length scale (positive value).
        This is the distance value at which edge formation
        probability equals ``1``.
    alpha : float
        Homophily value. It determines rate at which edge formation
        probability decreases/increases with the distance from the characteristic
        length scale `b`. Usually it should be set to a positive value.
    p_rewire : float
        Defualt probability of random edge rewiring when generating
        adjacency matrices. This is used to ensure small worldedness.
    directed : bool
        Should directed networks be generated by default.
    degseq : (N,) array_like, optional
        Default node degree sequence used in the configuration model.
    """
    def __init__(self, P, k, b, alpha, p_rewire=0.01, directed=False):
        """Initialization method."""
        self.P = P
        self.k = k
        self.b = b
        self.alpha = alpha
        self.p_rewire = p_rewire
        self.directed = directed
        self.degseq = None

    def __repr__(self):
        nm = self.__class__.__name__
        D = 'D' if self.directed else 'U'
        a = self.alpha
        p = self.p_rewire
        return f"<{nm} {D}{self.N} k={self.k} b={self.b:.2} a={a} p={p}>"

    @property
    def N(self):
        return self.P.shape[0]

    def sort_degseq(self, degseq):
        """Sort degseq accordingly to centrality of nodes in the social space."""
        w = self.P.sum(axis=1)
        return degseq[np.lexsort((w, w.argsort()[::-1]))]

    def set_degseq(self, degseq, sort=True):
        """Set default degree sequence.

        Parameters
        ----------
        sort : bool
            Should degree sequence be sorted to align with highest
            average edge formation probabilities per node.
        """
        degseq = degseq.copy()
        if sort and degseq is not None:
            degseq = self.sort_degseq(degseq)
        self.degseq = degseq

    @staticmethod
    def prob_measure(D, b, alpha):
        """Compute *SDA model* probability measure.

        Parameters
        ----------
        D : (N, N) array_like
            Distance matrix.
        b : float
            Characteristic length scale (positive value).
        alpha :
            Homophily value.
        """
        if b == 0:
            return np.zeros_like(D, dtype=float)
        P = 1 / (1 + (D/b)**alpha)
        np.fill_diagonal(P, 0)
        return P

    @classmethod
    def optim_b(cls, D, k, alpha, b_optim_min=0, b_optim_max=None,
                xtol=10**-12, **kwds):
        """Find optimal value of the characteristic length scale `b`.

        Parameters
        ----------
        D : (N, N) array_like
            Distance matrix.
        b : float
            Characteristic length scale (positive value).
        alpha :
            Homophily value.
        b_optim_min : float
            Lower bound of the interval used in optimizing `b`.
        b_optim_max : float, optional
            Uppe bound of the interval used in optimizing `b`.
            If ``None`` then defaults to ``D.max()``.
        **kwds :
            Keyword parameters passed to :py:func:`scipy.optimize.brentq`.
        """
        def b_optim(b):
            P = cls.prob_measure(D, b, alpha)
            Ek = P.sum(axis=1).mean()
            return k - Ek

        if b_optim_max is None:
            b_optim_max = D.max()
        b = toms748(b_optim, a=b_optim_min, b=b_optim_max, xtol=xtol, **kwds)
        return b

    @classmethod
    def from_dist_matrix(cls, D, k, alpha, p_rewire=0.01, directed=False, **kwds):
        """Constructor method based on a distance matrix.

        Parameters
        ----------
        D : (N, N) array_like
            Distance matrix.
        **kwds :
            Keyword parameters passed to `optim_b`.
        """
        b = cls.optim_b(D, k, alpha, **kwds)
        P = cls.prob_measure(D, b, alpha)
        return cls(P, k, b, alpha, p_rewire=p_rewire, directed=directed)

    @classmethod
    def from_weighted_dist_matrices(cls, k, alpha, dm, weights=None, p_rewire=0.01,
                                    directed=False, **kwds):
        """Constructor method based on a weighted sequence of distance matrices.

        Parameters
        ----------
        dm : (m,) sequence of (N, N) array_like
            Sequence of distance matrices.
        weights : (m,) array_like, optional
            Weights used for combining arrays.
            If ``None`` then defaults to equal weights.
        **kwds :
            Keyword parameters passed to `optim_b`.
        """
        P = None
        sum_w = 0
        loop = product(dm, (1,)) if weights is None else zip(dm, weights)
        for D, w in loop:
            b = cls.optim_b(D, k, alpha, **kwds)
            _P = cls.prob_measure(D, b, alpha)
            if P is None:
                P = _P*w
            else:
                P += _P*w
            sum_w += w
        P = P / sum_w
        return cls(P, k, b, alpha, p_rewire=p_rewire, directed=directed)

    def adjacency_matrix(self, sparse=True, p_rewire=None, directed=None):
        """Generate an adjacency matrix.

        Parameters
        ----------
        sparse : bool
            Should sparse matrix be used.
            If ``True`` then :py:class:`scipy.sparse.csr_matrix`
            is used.
        p_rewire : float, optional
            Random edge rewiring probability.
            If ``None`` then defaults to the class attribute value.
        directed : bool or None
            Should directed graph be generated.
            If ``None`` then class attribute is used.
        """
        if p_rewire is None:
            p_rewire = self.p_rewire
        if directed is None:
            directed = self.directed
        A = make_adjacency_matrix(self.P, directed=directed)
        if p_rewire > 0:
            A = rewire_edges(A, p=p_rewire, directed=directed, copy=False)
        if sparse:
            A = csr_matrix(A)
        return A

    def conf_model(self, simplify=True, sparse=True, p_rewire=None, directed=None,
                   p_malformed=10**-9):
        """Generate an adjacency matrix from the configuration model.

        Parameters
        ----------
        simplify : bool
            Should resulting adjacency matrix be simplified by removing
            self-loops and multiple edges, which are rare in larger graphs
            but sometimes can not be avoided.
        sparse : bool
            Should sparse matrix be used.
            If ``True`` then :py:class:`scipy.sparse.csr_matrix`
            is used.
        p_rewire : float, optional
            Random edge rewiring probability.
            If ``None`` then defaults to the class attribute value.
        directed : bool or None
            Should directed graph be generated.
            If ``None`` then class attribute is used.
        configuration : bool
            Should configuration model be used.
            If ``False`` then standard SDA network is generated.
        p_malformed : float
            Probability assigned to self-loops, multiple edges and impossible
            edges in the case of hard RGG (infinite `alpha`).
            This should be very small, but non-zero value.
        self_loop_p : float
            Probability of self-loop. This must be nonzero but preferably
            very small.
        multiple_edges_p : float
            Probability of generating multiple egdes. This must be nonzero
            but preferably very small.
        """
        # pylint: disable=too-many-branches
        # pylint: disable=too-many-statements

        # Argument checks and setup of control variables
        if p_rewire is None:
            p_rewire = self.p_rewire
        if directed is None:
            directed = self.directed
        degseq = self.degseq
        if degseq is None:
            raise ValueError("'degseq' must be set with 'set_degseq' method")
        if directed:
            out_stubs = degseq[:, 0].copy()
            in_stubs = degseq[:, 1].copy()
        else:
            in_stubs = out_stubs = degseq.copy()
        if not directed and degseq.sum() % 2 != 0:
            raise ValueError("'degseq' must be even for undirect graphs")
        if directed and in_stubs.sum() != out_stubs.sum():
            raise ValueError("sums of in and out degrees must be equal for directed graphs")
        if directed:
            n_stubs = in_stubs.sum() + out_stubs.sum()
        else:
            n_stubs = degseq.sum()
        # Configuration algorithm
        P = self.P.copy()
        np.fill_diagonal(P, p_malformed)
        # Assign non-zero probabilities to edges impossible due to homophily
        # i.e. in the case of hard RGG
        P[P == 0] = p_malformed
        # Assign zero probabilities to edges with no stubs
        idx = np.where(out_stubs == 0)
        P[idx, :] = 0
        if not directed:
            P[:, idx] = 0
        if directed:
            idx = np.where(in_stubs == 0)
            P[:, idx] = 0
        # Main configuration algorithm loop
        W = P.sum(axis=1)
        A = np.zeros(self.P.shape, dtype=int)
        while n_stubs > 0:
            i = random_integer(W/W.sum())
            j = random_integer(P[i, :]/P[i, :].sum())
            # Decrement and check out stubs
            out_stubs[i] -= 1
            if out_stubs[i] == 1:
                if not directed or in_stubs[i] == 0:
                    W[i] -= P[i, i]
                    P[i, i] = 0
            elif out_stubs[i] == 0:
                W[i] = 0
                P[i, :] = 0
                if not directed:
                    W -= P[:, i]
                    P[:, i] = 0
            # Decrement and check in stubs
            in_stubs[j] -= 1
            if in_stubs[j] == 1:
                if not directed or out_stubs[j] == 0:
                    W[j] -= P[j, j]
                    P[j, j] = 0
            elif in_stubs[j] == 0:
                W -= P[:, j]
                P[:, j] = 0
                if not directed:
                    W[j] = 0
                    P[j, :] = 0
            # Decrement total stubs and update adjacency matrix
            n_stubs -= 2
            A[i, j] += 1
            if P[i, j] > 0:
                W[i] += p_malformed - P[i, j]
                P[i, j] = p_malformed
            if not directed:
                A[j, i] += 1
                if P[j, i] > 0:
                    W[j] += p_malformed - P[j, i]
                    P[j, i] = p_malformed
        # Final processing
        if simplify:
            np.fill_diagonal(A, 0)
            A[A > 1] = 1
        if p_rewire > 0:
            A = rewire_edges(A, p=p_rewire, directed=directed, copy=False)
        if sparse:
            A = csr_matrix(A)
        return A

    def to_dict(self, P=False, am='normal', ds=False, **kwds):
        """Dump to ordered dictionary.

        Parameters
        ----------
        P : bool
            Should edge formation probability matrix be returned.
        am : {'normal', 'configuration'} or bool or None
            Return stadard adjacency matrix if ``normal`` or ``True``.
            Return configuration model adjacency matrix if ``configuration``.
            Return ``None`` if ``None`` or ``False``.
        ds : bool
            Should degree sequence for configuration model be returned.
        **kwds :
            Keyword arguments passed to the adjacency matrix method.

        Raises
        ------
        ValueError
            If `A` value is incorrect.
        """
        if not am:
            A = None
        elif am is True or am == 'normal':
            A = self.adjacency_matrix(**kwds)
        elif am == 'configuration':
            A = self.conf_model(**kwds)
        else:
            raise ValueError("incorrect 'am' value")
        _ds = kwds.get('degseq', self.degseq)
        return OrderedDict([
            ('P', self.P if P else None),
            ('A', A),
            ('N', self.N),
            ('k', self.k),
            ('b', self.b),
            ('alpha', self.alpha),
            ('p_rewire', self.p_rewire),
            ('directed', self.directed),
            ('degseq', _ds if ds else None)
        ])

In [3]:
"""Code for appendix A: draw example graphs from SDA and SDAC models."""
# pylint: disable=redefined-outer-name,no-member
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import networkx as nx
from networkx.algorithms import community
#import _


HERE = os.path.dirname(os.path.realpath('anibal/complex_path_centola/talaga-homophily-network.py'))
FIGS = os.path.join(HERE, 'figures')
if not os.path.exists(FIGS):
    os.makedirs(FIGS, exist_ok=True)

DPI = 150
FIGSIZE = (12, 12)
# PIXELS = 800

In [4]:

N = 400
NDIM = 2
CENTERS = 4
K = 15
ALPHA_SMALL = 2
ALPHA_LARGE = np.inf


In [5]:
# Routines --------------------------------------------------------------------

def make_graph(A):
    G = nx.from_numpy_array(A)
    communities = community.greedy_modularity_communities(G)
    for idx, comm in enumerate(communities, 1):
        for node_idx in comm:
            G.nodes[node_idx]['community'] = idx
            G.nodes[node_idx]['color'] = cm.Set1(idx-1)
    return G

def viz_space(ax, X, labels=None, edgecolors='#000000', linewidths=.6, **kwds):
    x, y = X[:, 0], X[:, 1]
    if labels is not None:
        labels = [ cm.Set1(x) for x in labels ]
    ax.scatter(x, y, c=labels, edgecolors=edgecolors, linewidths=linewidths,
               **kwds)
    ax.set_axisbelow(True)
    ax.grid(zorder=0, linestyle='--')
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    return ax

def viz_degseq(ax, degseq):
    ax.hist(degseq, color=cm.Set1(1), edgecolor='#000000', linewidth=1)
    ax.get_xaxis().set_ticks([])
    ax.get_yaxis().set_ticks([])
    return ax

def viz_sda_graph(ax, G, pos=None, with_labels=False, node_color=None,
                  node_border_color='#000000', node_border_width=.6,
                  edge_color='#000000', nodes_kws=None, edges_kws=None,
                  size_scaler=lambda x: np.sqrt(x)*2):
    if nodes_kws is None:
        nodes_kws = {}
    if edges_kws is None:
        edges_kws = {}
    if node_color is None:
        node_color = np.array([ n['color'] for n in G.nodes.values() ])
    node_size = np.array([ size_scaler(k) for k in G.degree() ])
    if pos is None:
        pos = nx.drawing.kamada_kawai_layout(G)
    nodes = \
        nx.draw_networkx_nodes(G, pos, node_color=node_color, node_size=node_size,
                               with_labels=with_labels, ax=ax, **nodes_kws)
    nodes.set_edgecolor(node_border_color)
    nodes.set_linewidth(node_border_width)
    nx.draw_networkx_edges(G, pos, edge_color=edge_color, alpha=.2,
                           ax=ax, **edges_kws)
    ax.axis('off')
    return ax

In [6]:
from sklearn.datasets import make_blobs

def simulate_uniform(N, ndim, a=0, b=1, **kwds):
    return np.random.uniform(a, b, (N, ndim)), None

def simulate_normal(N, ndim, mean=0, sd=1, **kwds):
    return np.random.normal(mean, sd, (N, ndim)), None

def simulate_lognormal(N, ndim, mean=0, sd=1, **kwds):
    return np.random.lognormal(mean, sd, (N, ndim)), None

def simulate_normal_clusters(N, ndim, centers=4, center_box=(-8, 8), **kwds):
    return make_blobs(N, ndim, centers=centers, center_box=center_box, **kwds)

In [7]:

# SDA model -------------------------------------------------------------------

# Set random seed
np.random.seed(44)

uniform, __ = simulate_uniform(N, NDIM)
lognormal, __ = simulate_lognormal(N, NDIM)
clusters, clusters_labs = simulate_normal_clusters(N, NDIM, CENTERS)

# Make plot
plt.close()
fig = plt.figure(figsize=FIGSIZE)
idx = 0
for X, labs in ((uniform, None), (clusters, clusters_labs), (lognormal, None)):
    D = make_dist_matrix(X, euclidean_dist, symmetric=True).astype(np.float32)
    for alpha in (None, ALPHA_SMALL, ALPHA_LARGE):
        idx += 1
        ax = plt.subplot(3, 3, idx)
        if alpha is None:
            ax = viz_space(ax, X, labs)
        else:
            sda = SDA.from_dist_matrix(D, alpha=alpha, k=K, directed=False)
            A = sda.adjacency_matrix(sparse=False)
            G = make_graph(A)
            ax = viz_sda_graph(ax, G)
        if idx <= 3:
            if alpha is None:
                title = '(a)'
            elif alpha == ALPHA_SMALL:
                title = '(b)'
            else:
                title = '(c)'
            ax.set_title(title)
plt.savefig(os.path.join(FIGS, 'fig-A-1.png'), dpi=DPI, bbox_inches='tight')

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1mNameError: name 'Lp_dist' is not defined[0m
[0m[1mDuring: resolving callee type: type(CPUDispatcher(<function euclidean_dist at 0x7f3bfff7a950>))[0m
[0m[1mDuring: typing of call at /scratch/local/u6060602/1498443/ipykernel_2370645/964245765.py (30)
[0m
[0m[1mDuring: resolving callee type: type(CPUDispatcher(<function euclidean_dist at 0x7f3bfff7a950>))[0m
[0m[1mDuring: typing of call at /scratch/local/u6060602/1498443/ipykernel_2370645/964245765.py (34)
[0m
[0m[1mDuring: resolving callee type: type(CPUDispatcher(<function euclidean_dist at 0x7f3bfff7a950>))[0m
[0m[1mDuring: typing of call at /scratch/local/u6060602/1498443/ipykernel_2370645/964245765.py (30)
[0m
[1m
File "../../../../../../../scratch/local/u6060602/1498443/ipykernel_2370645/964245765.py", line 30:[0m
[1m<source missing, REPL/exec in use?>[0

<Figure size 864x864 with 0 Axes>